## 1. Import & Configuration


In [33]:
from serpapi import GoogleSearch
from youtube_transcript_api import YouTubeTranscriptApi 
from concurrent.futures import ThreadPoolExecutor, as_completed 
import trafilatura 
import pandas as pd
from dotenv import load_dotenv
import os 
import re
import time

load_dotenv() 
SERP_API_KEY = os.getenv("SERPAPI_KEY")
# Max Number of Article to Scrape
MAX_NUM_RESULTS = 10

## 2. SerAPI - Collection article & Video URLs


In [34]:

from pandas._libs.tslibs import Tick


def collect_search_results(queries: list[str] , num_results: int=MAX_NUM_RESULTS) -> tuple[pd.DataFrame, list[dict]]:
    """
    Search Google for each query and collect
    - organic results : article (title , snippet, url , source, date)
    - incline videos : YouTube videos (title, channel , url)

    Returns : 
      articles_df : DataFrame of all metadata.
      raw_results : list of raw SerpAPI result dicts (just for incepection).
    """
    article_rows = []
    video_rows = [] 
    raw_results = [] 

    for query in queries:
        print(f"\n Searching : {query}") 
        params = {
            "engine" : "google" ,
            "q" : query,
            "google_domain" : "google.com" ,
            "hl" : "en" ,
            "gl" : "us" ,
            "api_key" : SERP_API_KEY
        }
        search = GoogleSearch(params) 
        results = search.get_dict()
        raw_results.append({"query" : query , "results":results}) 
        # Just for checking
        print(f"Total Number of Keys in the search results : {results.keys()}")
        # --- Organic arcticle -----
        organic = results.get("organic_results" , []) 
        if not organic:
            print(f"No orgainc results for : {query}") 
        else:
            for art in organic:
                article_rows.append({
                    "query" : query, 
                    "title" : art.get("title") ,
                    "snippet" : art.get("snippet") ,
                    "source" : art.get("source") ,
                    "date" : art.get("date") ,
                    "url" : art.get("link") ,
                    "type" : "article"
                })
                   
            print(f"{len(organic[:num_results])} articles_collected") 

            # -- Incline Video --
            videos = results.get("inline_videos" , [] ) 
            if videos:
                for vid in organic:
                    video_rows.append({
                        "query" : query, 
                        "title" : vid.get("title") ,
                        "snippet" : None ,
                        "source" : vid.get("channel") ,
                        "date" : None ,
                        "url" : vid.get("link") ,
                        "type" : "video"
                    })
                print(f"{len(videos)} video collected")
            else:
                print(f"No videos results on this {query}")

            # Combine all into single dataframe.
    all_rows = article_rows + video_rows
    df = pd.DataFrame(all_rows) 

    if df.empty:
        print("\n No results collected accross all queries") 
        return df , raw_results 
    df = df.drop_duplicates(subset="url").reset_index(drop=True) 

    return df,  raw_results 
            
            


In [35]:
from gc import collect


queries = ["AI intellectual property" , "CopyRight Generative AI"] 
df , raw_results = collect_search_results(queries)
df.head()


 Searching : AI intellectual property
Total Number of Keys in the search results : dict_keys(['search_metadata', 'search_parameters', 'search_information', 'related_questions', 'ai_overview', 'organic_results', 'perspectives', 'related_searches', 'pagination', 'serpapi_pagination'])
8 articles_collected
No videos results on this AI intellectual property

 Searching : CopyRight Generative AI
Total Number of Keys in the search results : dict_keys(['search_metadata', 'search_parameters', 'search_information', 'related_questions', 'ai_overview', 'organic_results', 'related_searches', 'pagination', 'serpapi_pagination'])
9 articles_collected
No videos results on this CopyRight Generative AI


,query,title,snippet,source,date,url,type
0,AI intellectual property,‎Google Gemini,"Meet Gemini, Google's AI assistant. Get help w...",‎Google Gemini,NaN,https://gemini.google.com/,article
1,AI intellectual property,Artificial intelligence,Artificial intelligence (AI) is the capability...,Wikipedia,NaN,https://en.wikipedia.org/wiki/Artificial_intel...,article
2,AI intellectual property,Perplexity AI,Perplexity is a free AI-powered answer engine ...,Perplexity,NaN,https://www.perplexity.ai/,article
3,AI intellectual property,"ChatGPT: Chat, Work, Create & Code with AI","Use ChatGPT to answer questions, write, create...",ChatGPT,NaN,https://chatgpt.com/,article
4,AI intellectual property,What is Artificial Intelligence (AI)?,AI is a field of computer science focused on c...,Google Cloud,NaN,https://cloud.google.com/learn/what-is-artific...,article


In [36]:
raw_results[1]['results']['organic_results']



[{'position': 1,
  'title': 'Copyright and Artificial Intelligence | U.S. Copyright Office',
  'link': 'https://www.copyright.gov/ai/',
  'redirect_link': 'https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=CAESWAHrOzAVwGnmTnfNjZz4bim_bK7YwuHYlnsWGLScV5qLZ9jxymY4lxjBr44g3Y3R5awZfVcWnOYrATC0g-NFOrxnKi8Cgu-yVM670mFj-pnrgpFWfzkoyO8&ved=2ahUKEwjy4Y3MvNCXAxUcK1kFHVyaBD4QFnoECBsQAQ&uoh=1',
  'displayed_link': 'https://www.copyright.gov › ai',
  'favicon': 'https://serpapi.com/images/i/iVBORw0KGgoAAAANSUhEUgAAABwAAAAcCAAAAABXZoBIAAABUElEQVR4AX2TIayDMBCGT2GqmxCSJiSY4pP5yYnpyUn83OQUZj7BombRyPqkEj2HQmHvf3RtG7K87EvAfOT-Xu8gJBY79P1gVySi5OFMQha5zLKr2UsGTC2q2qMrOk7gIBl4kK4jWm-6R5KNiKYUJJUkQW2QuEdXUTPODJ7HM_VgJ0cKLj9NiAw0YZOsQp68YYc5ONnLEHfZO8bdbPLge9DZjA9rP34aWa6gd0hUD3hOgiwczUpjqCpsyKJavuB4TdQVXtLq5UPVqjWObqRW-cgCnqbcIgQ5hn-lPt43bjaW1bGs-7gKXaUDZVM6UF02QcZWig6ei1Bk4sgOWvsxspfc3UY47O76ita7-JrN_uLpFdSH9QoQI42M2jU5m01fwy7U0y6M9T1cqIeXaKLVhSCZbw89EeSvBQPCamovS7eaO8ng4URC5t9LnVjM9-_wB5iH1

## Scrapping texts from the articles and videos

In [45]:
# Define scrapers


def _scrape_url(url:str) -> dict:
    """Scrape clean article text from a URL using Trafilatura""" 
    try:
        downloaded = trafilatura.fetch_url(url) 
        if not downloaded:
            return {"url" : url , "full_text" : None , "status" : "failed_download"} 
        
        text = trafilatura.extract(
            downloaded,
            include_comments = False,
            include_tables = True ,
            no_fallback = False, 
            favor_precision = False, #prefer less but correct extraction
            deduplicate = True
        )
        return {
            "url" : url ,
            "full_text" : text.strip() if text else None,
            "status" : "success" if text else "failed_extraction"
        }
    except Exception as e :
        return {"url" : url , "full_text" : None , "status" : f"error : {str(e)}"} 



In [38]:
def _get_youtube_id(url:str) -> str  | None:
    """Extract the  YouTube VideoId from the URL""" 
    patterns = [
        r"youtube\.com/watch\?v=([a-zA-Z0-9_-]{11})" ,
        r"youtube\.com/shorts/([a-zA-Z0-9_-]{11})" ,
        r"youtu\.be/([a-zA-Z0-9_-]{11})" 
    ] 
    for pattern in patterns:
        match = re.search(pattern,url) 
        if match:
            return match.group(1)  # # "dQw4w9WgXcQ" → only what's inside ( )
    return None


In [39]:
def _get_transcript(video_id : str) -> str | None:
    """Fetch YouTube transcript from video ID."""  
    try:
        transcript = YouTubeTranscriptApi.get_transcript(
            video_id , languages = ["en" , "en-US" , "en-GB"] 
        )
        return " ".join([t["text"] for t in transcript])
    except Exception as e: 
        print(f"Error : {str(e)}")
        return None         


In [42]:

def enrich_search_results(df: pd.DataFrame, max_workers: int = 5, delay : float = 1.0) -> pd.DataFrame:
    """
    Enrich a DataFrame from collect_search_results() with full text content.
    - Rows with type='article' --> scraped via Trafilatura 
    - Rows with type='video' --> transcript via YouTube Transcript API
    Args:
     df : DataFrame returned by collect_articles() 
     max_workers : concurrent threads for article scraping
     delay : seconds between requests per thread
    
    Returns:
    Enriched DataFrame with added columns:
    - full_text : scrapped article text OR video transcript
    - video_id : YouTube video ID (videos only) 
    - status : success / failed_download / no_transcript / etc. 
    """ 
    if df.empty:
        print("Empty DataFrame - nothing to enrich.")
        return df 
    df = df.copy() 
    df["full_text"] = None
    df["video_id"] = None
    df["status"] = "pending" 

    # --- Split by type ---  
    article_df = df[df["type"] == "article"].copy()
    video_df   = df[df["type"] == "video"].copy() 
    # Alternative way of splitting using group by.
    # dfs = {t: g.copy() for t, g in df.groupby("type")}
    # article_df = dfs["article"]
    # video_df   = dfs["video"]

    # -- Scrape arcticles -- 
    if not article_df.empty:
        print(f"\n Scraping {len(article_df)} articles...") 
        url_to_result = {}
        def fetch_with_delay(url):
            time.sleep(delay) 
            return _scrape_url(url)  
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(fetch_with_delay, row["url"]) : idx for idx, row in article_df.iterrows()
            }
            # In this futures it is dict comphrensive 
            # A Future is a placeholder for a result that isn't ready yet. 
            # It's like a receipt or order token: you place an order, get a token right away, and collect the food when it's ready.
            for i , future in enumerate(as_completed(futures)):
                result = future.result() 
                url_to_result[result["url"]] = result  
                icon = "✅" if result["status"] == "success" else "❌" 
                source = df.loc[futures[future],"source"] or "Unknown" 
                print(f" {icon} [{i+1}/{len(article_df)}] " 
                f"{source} — {result['status']}"
                )
            # Map results back 
            for idx , row in article_df.iterrows():
                r = url_to_result.get(row["url"] , {}) 
                df.at[idx , "full_text"] = r.get("full_text")
                df.at[idx , "status"] = r.get("status" , "unknown") 

            success = sum(1 for r in url_to_result.values() if r["status"] == "success") 
            print(f"\n  📊 Articles: {success}/{len(article_df)} "
              f"scraped successfully") 

        # -- Fetch Video Transcript --
        if not video_df.empty:
            print(f"\n🎬 Fetching transcripts for "
              f"{len(video_df)} videos...")
            for idx , row in video_df.iterrows():
                url = row['url'] 
                vid_id = _get_youtube_id(url) 
                transcript = None 
                if vid_id:
                    transcript = _get_transcript(vid_id)
                status = "success" if transcript else "no_transcript" 
                icon   = "✅" if transcript else "❌" 

                df.at[idx , "video_id"] = vid_id
                df.at[idx , "full_text"] = transcript 
                df.at[idx , "status"] = status
                title = (row["title"] or "")[:50]
                print(f"  {icon} {title}... [{status}]")

            success = sum(1 for r in video_df.index if df.at[idx , 'status'] == 'success') 

            print(f"\n  📊 Videos: {success}/{len(video_df)} "
              f"transcripts fetched")

        return df


In [43]:
df.head()

,query,title,snippet,source,date,url,type
0,AI intellectual property,‎Google Gemini,"Meet Gemini, Google's AI assistant. Get help w...",‎Google Gemini,NaN,https://gemini.google.com/,article
1,AI intellectual property,Artificial intelligence,Artificial intelligence (AI) is the capability...,Wikipedia,NaN,https://en.wikipedia.org/wiki/Artificial_intel...,article
2,AI intellectual property,Perplexity AI,Perplexity is a free AI-powered answer engine ...,Perplexity,NaN,https://www.perplexity.ai/,article
3,AI intellectual property,"ChatGPT: Chat, Work, Create & Code with AI","Use ChatGPT to answer questions, write, create...",ChatGPT,NaN,https://chatgpt.com/,article
4,AI intellectual property,What is Artificial Intelligence (AI)?,AI is a field of computer science focused on c...,Google Cloud,NaN,https://cloud.google.com/learn/what-is-artific...,article


In [46]:
df= enrich_search_results(df)
df.head()



 Scraping 17 articles...
 ❌ [1/17] Perplexity — failed_extraction
 ✅ [2/17] ‎Google Gemini — success
 ✅ [3/17] ChatGPT — success
 ✅ [4/17] OpenAI — success
 ✅ [5/17] Copyright Office (.gov) — success
 ✅ [6/17] ai.google — success
 ✅ [7/17] National Institute of Standards and Technology (.gov) — success
 ✅ [8/17] Wikipedia — success
 ❌ [9/17] ScienceDirect.com — failed_download
 ✅ [10/17] University of South Florida — success
 ❌ [11/17] Reddit · r/Screenwriting — failed_extraction
 ✅ [12/17] University of Southern California — success
 ✅ [13/17] The University of Virginia — success
 ✅ [14/17] Issues in Science and Technology — success
 ❌ [15/17] Congress.gov — failed_download
 ✅ [16/17] The Regulatory Review — success
 ✅ [17/17] Google Cloud — success

  📊 Articles: 13/17 scraped successfully


,query,title,snippet,source,date,url,type,full_text,video_id,status
0,AI intellectual property,‎Google Gemini,"Meet Gemini, Google's AI assistant. Get help w...",‎Google Gemini,NaN,https://gemini.google.com/,article,plus\nmic\narrow_upward\nSign in,None,success
1,AI intellectual property,Artificial intelligence,Artificial intelligence (AI) is the capability...,Wikipedia,NaN,https://en.wikipedia.org/wiki/Artificial_intel...,article,Artificial intelligence\n| Part of a series on...,None,success
2,AI intellectual property,Perplexity AI,Perplexity is a free AI-powered answer engine ...,Perplexity,NaN,https://www.perplexity.ai/,article,None,None,failed_extraction
3,AI intellectual property,"ChatGPT: Chat, Work, Create & Code with AI","Use ChatGPT to answer questions, write, create...",ChatGPT,NaN,https://chatgpt.com/,article,Ready when you are.\nChatGPT is AI. By using i...,None,success
4,AI intellectual property,What is Artificial Intelligence (AI)?,AI is a field of computer science focused on c...,Google Cloud,NaN,https://cloud.google.com/learn/what-is-artific...,article,Ever wonder how your phone can recognize your ...,None,success


In [ ]:
# -- Filter to only successfully scraped content -- 
df_clean = df[df["status"] == "success"].reset_index(drop=True) 
# -- Inspect results -- 
df_clean.head() 

,query,title,snippet,source,date,url,type,full_text,video_id,status
0,AI intellectual property,‎Google Gemini,"Meet Gemini, Google's AI assistant. Get help w...",‎Google Gemini,NaN,https://gemini.google.com/,article,plus\nmic\narrow_upward\nSign in,None,success
1,AI intellectual property,Artificial intelligence,Artificial intelligence (AI) is the capability...,Wikipedia,NaN,https://en.wikipedia.org/wiki/Artificial_intel...,article,Artificial intelligence\n| Part of a series on...,None,success
2,AI intellectual property,"ChatGPT: Chat, Work, Create & Code with AI","Use ChatGPT to answer questions, write, create...",ChatGPT,NaN,https://chatgpt.com/,article,Ready when you are.\nChatGPT is AI. By using i...,None,success
3,AI intellectual property,What is Artificial Intelligence (AI)?,AI is a field of computer science focused on c...,Google Cloud,NaN,https://cloud.google.com/learn/what-is-artific...,article,Ever wonder how your phone can recognize your ...,None,success
4,AI intellectual property,OpenAI | Research & Deployment,We believe our research will eventually lead t...,OpenAI,NaN,https://openai.com/,article,What can I help with? Message ChatGPT Talk wit...,None,success


In [53]:
print(df_clean.info())

<class 'pandas.DataFrame'>
RangeIndex: 13 entries, 0 to 12
Data columns (total 10 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   query      13 non-null     str   
 1   title      13 non-null     str   
 2   snippet    13 non-null     str   
 3   source     13 non-null     str   
 4   date       4 non-null      str   
 5   url        13 non-null     str   
 6   type       13 non-null     str   
 7   full_text  13 non-null     object
 8   video_id   0 non-null      object
 9   status     13 non-null     str   
dtypes: object(2), str(8)
memory usage: 1.1+ KB
None
